# Experiment: Combined Experiment Overview

Loads the exported exact-match plotting data from the four main experiment notebooks and renders a single combined figure with one row per experiment.


In [ ]:
from __future__ import annotations

import sys

import pandas as pd
import pyrootutils

PROJECT_ROOT = pyrootutils.find_root(indicator=".project-root")
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CACHE_DIR = NOTEBOOKS_DIR / "cache" / "combined-exp"
FIGURES_DIR = PROJECT_ROOT / "paper" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import aesthetics as aes  # noqa: E402

sns = aes.sns
plt = aes.plt

CACHE_DIR

## Load cached plot data

Run the source notebooks first so their export cells populate `notebooks/cache/combined-exp/`.


In [ ]:
AGREEMENT_CONDITION_ORDER = [
    "NoAgr → NoAgr",
    "Agr → NoAgr",
    "Agr → Agr",
    "NoAgr → Agr",
]

PALETTE_AGREEMENT = aes.darken(
    {
        "NoAgr → NoAgr": sns.color_palette("Dark2", n_colors=4)[0],
        "Agr → NoAgr": sns.color_palette("Dark2", n_colors=4)[1],
        "Agr → Agr": sns.color_palette("Dark2", n_colors=4)[2],
        "NoAgr → Agr": sns.color_palette("Dark2", n_colors=4)[3],
    },
    by=0.25,
)

ORTHOGRAPHY_ORDER = [
    "Latin → Latin",
    "Latin → Latin (diacritics)",
    "Latin → Cyrillic",
    "Latin → Hebrew",
    "Latin → Hebrew (pointed)",
]

PALETTE_ORTHOGRAPHY_LARGE = aes.darken(
    {
        "Latin → Latin": sns.color_palette("Reds", n_colors=7)[1],
        "Latin → Latin (diacritics)": sns.color_palette("Reds", n_colors=7)[3],
        "Latin → Cyrillic": sns.color_palette("Greens", n_colors=7)[2],
        "Latin → Hebrew (pointed)": sns.color_palette("Blues", n_colors=7)[3],
        "Latin → Hebrew": sns.color_palette("Blues", n_colors=7)[5],
    },
    by=0.15,
)

WORD_ORDER_PLOT_ORDER = ["SVO", "SOV", "OVS"]
WORD_ORDER_PLOT_LABELS = {
    "SVO": "SVO → SVO",
    "SOV": "SVO → SOV",
    "OVS": "SVO → OVS",
}

EXPERIMENTS = [
    {
        "slug": "size",
        "display_name": "Size (all models)",
        "size_file": "size_grammar_size.csv",
        "length_file": "size_string_length.csv",
        "size_x": "size",
        "length_x": "input_words_binned_quant_num",
        "hue": "model_display_name",
        "style": "model_display_name",
        "order": None,
        "palette": None,
        "legend_labels": None,
        "legend_cols": 2,
    },
    {
        "slug": "wordorder_large",
        "display_name": "Word order (gpt-5)",
        "size_file": "wordorder_large_grammar_size.csv",
        "length_file": "wordorder_large_string_length.csv",
        "size_x": "size",
        "length_x": "input_length_quintile_mid",
        "hue": "target_word_order",
        "style": "target_word_order",
        "order": WORD_ORDER_PLOT_ORDER,
        "palette": aes.PALETTE_WORDORDER,
        "legend_labels": WORD_ORDER_PLOT_LABELS,
        "legend_cols": 1,
    },
    {
        "slug": "agreement",
        "display_name": "Morphology (gpt-5)",
        "size_file": "agreement_grammar_size.csv",
        "length_file": "agreement_string_length.csv",
        "size_x": "grammar_size",
        "length_x": "input_length_quintile_mid",
        "hue": "agreement_condition",
        "style": "agreement_condition",
        "order": AGREEMENT_CONDITION_ORDER,
        "palette": PALETTE_AGREEMENT,
        "legend_labels": None,
        "legend_cols": 2,
    },
    {
        "slug": "orthography_large",
        "display_name": "Orthography (gpt-5)",
        "size_file": "orthography_large_grammar_size.csv",
        "length_file": "orthography_large_string_length.csv",
        "size_x": "grammar_size",
        "length_x": "input_length_quintile_mid",
        "hue": "target_orthography",
        "style": "target_orthography",
        "order": ORTHOGRAPHY_ORDER,
        "palette": PALETTE_ORTHOGRAPHY_LARGE,
        "legend_labels": None,
        "legend_cols": 1,
    },
]

missing_files = []
loaded = {}

for config in EXPERIMENTS:
    size_path = CACHE_DIR / config["size_file"]
    length_path = CACHE_DIR / config["length_file"]
    if not size_path.exists():
        missing_files.append(size_path)
    if not length_path.exists():
        missing_files.append(length_path)
    if size_path.exists() and length_path.exists():
        loaded[config["slug"]] = {
            "size": pd.read_csv(size_path),
            "length": pd.read_csv(length_path),
        }

if missing_files:
    missing_str = "\n".join(f"- {path}" for path in missing_files)
    message = (
        "Missing combined-exp inputs. Run the export cells in "
        "the source notebooks first:\n"
        f"{missing_str}"
    )
    raise FileNotFoundError(message)

for config in EXPERIMENTS:
    if config["slug"] == "size":
        size_df = loaded[config["slug"]]["size"]
        model_order = aes.ordered_models(size_df["model_name"].dropna().unique())
        config["order"] = [
            aes.MODEL_DISPLAY_NAMES.get(model, model) for model in model_order
        ]
        config["palette"] = {
            aes.MODEL_DISPLAY_NAMES.get(model, model): aes.PALETTE_MODELS[model]
            for model in model_order
            if model in aes.PALETTE_MODELS
        }

{
    slug: {panel: df.shape for panel, df in panels.items()}
    for slug, panels in loaded.items()
}

In [ ]:
fig, axes = plt.subplots(
    len(EXPERIMENTS),
    2,
    figsize=(aes.COLM_PAPER_WIDTH_IN, len(EXPERIMENTS) * aes.FIG_HEIGHT_SINGLE_ROW_IN),
    sharex="col",
)
axes = axes.reshape(len(EXPERIMENTS), 2)


def _format_row_legend(ax, config):
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return
    if config["order"] is not None:
        label_to_handle = dict(zip(labels, handles))
        ordered_labels = [
            label for label in config["order"] if label in label_to_handle
        ]
        handles = [label_to_handle[label] for label in ordered_labels]
        labels = ordered_labels
    if config["legend_labels"] is not None:
        labels = [config["legend_labels"].get(label, label) for label in labels]
    legend = ax.legend(
        handles,
        labels,
        title=config["display_name"],
        bbox_to_anchor=(1.02, 1.1),
        loc="upper left",
        borderaxespad=0,
        frameon=False,
        alignment="left",
        columnspacing=0.8,
        ncol=config["legend_cols"],
    )
    legend.get_title().set_fontweight("bold")
    # legend.get_title().set_horizontalalignment("left")


for row_idx, config in enumerate(EXPERIMENTS):
    size_df = loaded[config["slug"]]["size"].copy()
    length_df = loaded[config["slug"]]["length"].copy()
    left_ax = axes[row_idx, 0]
    right_ax = axes[row_idx, 1]

    sns.lineplot(
        data=size_df,
        x=config["size_x"],
        y="match_value",
        hue=config["hue"],
        style=config["style"],
        hue_order=config["order"],
        style_order=config["order"],
        palette=config["palette"],
        markers=True,
        markersize=7,
        linewidth=2,
        errorbar="ci",
        ax=left_ax,
    )
    sns.lineplot(
        data=length_df,
        x=config["length_x"],
        y="match_value",
        hue=config["hue"],
        style=config["style"],
        hue_order=config["order"],
        style_order=config["order"],
        palette=config["palette"],
        markers=True,
        markersize=7,
        linewidth=2,
        errorbar="ci",
        ax=right_ax,
    )

    for ax in (left_ax, right_ax):
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    left_ax.set_ylabel("Accuracy")
    right_ax.set_ylabel("")
    right_ax.yaxis.set_ticklabels([])

    left_legend = left_ax.get_legend()
    if left_legend is not None:
        left_legend.remove()
    _format_row_legend(right_ax, config)

    left_ax.set_title(None)
    right_ax.set_title(None)

    if row_idx < len(EXPERIMENTS) - 1:
        left_ax.set_xlabel("")
        right_ax.set_xlabel("")
    else:
        left_ax.set_xlabel("Grammar size")
        right_ax.set_xlabel("String length (binned into quintiles)")

    left_ax.xaxis.set_major_formatter(aes.KMB_FORMATTER)
    right_ax.xaxis.set_major_formatter(aes.NICE_FORMATTER)

axes[0, 0].set_xlim(0, 10_500)
axes[0, 1].set_xlim(5, 45)
axes[-1, 0].set_xticks([0, 2_500, 5_000, 7_500, 10_000])
axes[-1, 1].set_xticks([10, 20, 30, 40])

plt.subplots_adjust(left=0, bottom=0, right=1, top=1, hspace=0.3, wspace=0.1)
aes.save_figure(FIGURES_DIR / "combined_experiment_overview", fig=fig)
plt.show()